In [1]:
# torchvision과 torchaudio를 추가하여 torch 버전과 호환되도록 함께 업데이트합니다.
!pip -q install -U "torch>=2.2,<3.0" "torchvision" "torchaudio" "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"

import torch, transformers, datasets, evaluate
import numpy as np

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 136.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.8 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128 | CUDA: True
Transformers: 5.2.0 | Datasets: 4.5.0
Using device: cuda


In [2]:
from datasets import load_dataset
beans=load_dataset("beans")
print(beans)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})


In [5]:
beans['train'].features

{'image_file_path': Value('string'),
 'image': Image(mode=None, decode=True),
 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}

In [7]:
beans['train'].features

{'image_file_path': Value('string'),
 'image': Image(mode=None, decode=True),
 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}

In [9]:
key = 'labels' if 'labels' in beans['train'].features else 'label'
beans['train'].features[key].names

['angular_leaf_spot', 'bean_rust', 'healthy']

In [12]:
from transformers import AutoImageProcessor, ViTForImageClassification
import torch

# 모델 및 프로세서 설정
MODEL = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(MODEL)

# 라벨 매핑 설정
key = 'labels' if 'labels' in beans['train'].features else 'label'
names = beans['train'].features[key].names
id2label = {i: n for i, n in enumerate(names)}
label2id = {n: i for i, n in enumerate(names)}

# 모델 로드
model = ViTForImageClassification.from_pretrained(
    MODEL,
    num_labels=len(names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
    # 크기가 다른 헤드(layer) 무시 >> 재초기화
).to(device)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [14]:
model.classifier.out_features # 분류기 크기: 3개

3

In [16]:
import os

num_proc = os.cpu_count()
num_proc

def transform(ex):
    # 입력이 PIL 이미지 리스트 (batched=True)
    # processor 가 알아서 resize, 정규화 수행, tensor 변환
    inputs = processor(images=ex['image'], return_tensors='pt')
    # 이미지 리스트를 프로세서에 전달

    # 결과 저장
    ex['pixel_values'] = inputs['pixel_values']
    # pixel_values : 4차원 텐서로 출력 (b, c, h, w)
    # (입력된 이미지의) 픽셀 값만 추출 >> 배치에 추가
    return ex

# 먼저 map 전처리 수행 (병렬 처리 추가)
# remove_columns 사용, 원본 'image' 컬럼 제거 (메모리 절약)

beans = beans.map(
        transform,
        batched=True,
        remove_columns=['image'], # 학습에 필요 없는 원본 이미지 컬럼 삭제
        num_proc = num_proc,      # 핵심 : 멀티 프로세싱(속도 향상)
    )


# format 설정
# Hugging Face Trainer 쓰면 이 부분 생략 >> 대신, DataCollater 가 처리하게 함
beans.set_format('torch')

Map (num_proc=8):   0%|          | 0/1034 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/133 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/128 [00:00<?, ? examples/s]

In [17]:
def keep(split):
    cols = ['pixel_values', key]
    # pixel_values 이미지 데이터, key: 라벨
    # 즉, cols 는 이미지데이터, 라벨 을 남기겠다.
    # split : train, val, test
    return beans[split].remove_columns([c for c in beans[split].column_names if c not in cols])
    # 지울 컬럼리스트 = 전체 컬럼 - 남길 컬럼

In [18]:
train, val, test = keep('train'), keep('validation'), keep('test')

In [22]:
from transformers import TrainingArguments, Trainer, DefaultDataCollator
import evaluate
import numpy as np
import torch

# 정확도 지표 로드
acc = evaluate.load("accuracy")

# 평가 계산 함수 정의
def metrics(p):
    predictions, labels = p
    pred = np.argmax(predictions, axis=1)
    return {'accuracy': acc.compute(predictions=pred, references=labels)['accuracy']}

# TrainingArguments 설정
args = TrainingArguments(
        output_dir='/content/vit_beans',
        eval_strategy='epoch',
        save_strategy='epoch',
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=torch.cuda.is_available(),
        report_to='none',
        remove_unused_columns=False    # 이미지 데이터셋 컬럼 유지 위해 권장
    )

# Trainer 초기화
# tokenizer=processor >> 이것 대신에 data_collater 명시하는 게 정석임
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        data_collator=DefaultDataCollator(), # 텐서 배치를 위한 Collater
        compute_metrics=metrics
    )

# DefaultDataCollator()
# 각 개별 샘플들을 하나씩 깔끔하게 배치(batch) tensor 로 묶어주는 역할

# 학습 시작
trainer.train()

print(trainer.evaluate(test))

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.076690,0.977444
2,No log,0.036492,0.984962
3,No log,0.030351,0.984962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.07339537888765335, 'eval_accuracy': 0.96875, 'eval_runtime': 1.2342, 'eval_samples_per_second': 103.714, 'eval_steps_per_second': 3.241, 'epoch': 3.0}


In [25]:
import torch

for i in [0,1]:
    ex = beans['test'][i]

    # 입력 데이터 처리
    input_tensor = ex['pixel_values'].clone().detach()
    inputs = input_tensor.unsqueeze(0).to(model.device)

    # 모델 예측
    with torch.no_grad():
        logits = model(inputs).logits
        pred = logits.argmax(-1).item() # 정수변환

    # 정답 라벨 가져오기
    label_key = 'labels' if 'labels' in ex else 'label'

    true_label_id = ex[label_key].item() #.item() 사용하면 tensor(0) >> 0(정수)

    # 결과 출력
    print(f'[{i} 예측: {model.config.id2label[pred]} | {model.config.id2label[true_label_id]}]')



[0 예측: angular_leaf_spot | angular_leaf_spot]
[1 예측: angular_leaf_spot | angular_leaf_spot]
